# Part 1: Data Preprocessing
Parse BioScope corpus, fetch PubMed abstracts, build 60-sentence stimulus set.

**Output:** `final_stimulus_dataset.csv`

## What this notebook does
Parses the BioScope corpus (11,871 biomedical sentences annotated for hedging), applies selection criteria, fetches PubMed abstracts, and produces a balanced 60-sentence stimulus dataset.
## How to run
1. Download `abstracts.xml` from the `bioscope/` folder in the GitHub repo
2. Run each cell in order from top to bottom
3. When the first code cell asks you to upload a file, upload `abstracts.xml`
4. The notebook produces `final_stimulus_dataset.csv` which downloads automatically at the end
5. Save that CSV file - you need it for Part 2
## What happens inside
- Step 1: Parse BioScope XML, extract 11,871 sentences with hedge/negation annotations
- Step 2: Classify each sentence as ASSERTIVE (no speculation cues) or UNCERTAIN (has speculation cues)
- Step 3: Show corpus statistics and top hedge cues
- Step 4: Filter by token length (15-60), remove anaphoric starts, prefer strong hedge cues
- Step 5: Select 20 UNCERTAIN + 20 ASSERTIVE from BioScope (Source 1)
- Step 6: Fetch 20 more sentences from PubMed physics/CS/engineering (Source 2)
- Step 7: Combine into final 60 sentences (30 UNCERTAIN, 30 ASSERTIVE)
- Step 8: Save and download CSV
## No API key needed for this notebook

In [1]:
# Upload abstracts.xml from bioscope/ folder
from google.colab import files
uploaded = files.upload()  # upload abstracts.xml

Saving abstracts.xml to abstracts (1).xml


In [2]:
import xml.etree.ElementTree as ET
import re, json, csv, random
from collections import Counter

random.seed(42)

# Parse BioScope XML
with open('abstracts.xml', 'r', encoding='utf-8') as f:
    raw_xml = f.read()
raw_xml = re.sub(r'<!DOCTYPE[^>]+>', '', raw_xml)
root = ET.fromstring(raw_xml)

def extract_text_and_cues(sent_elem):
    plain = ' '.join(ET.tostring(sent_elem, encoding='unicode', method='text').split())
    spec_cues = [c.text.strip() for c in sent_elem.iter('cue') if c.get('type') == 'speculation' and c.text]
    neg_cues = [c.text.strip() for c in sent_elem.iter('cue') if c.get('type') == 'negation' and c.text]
    return plain, spec_cues, neg_cues

all_sentences = []
for doc in root.iter('Document'):
    doc_id_elem = doc.find('DocID')
    doc_id = doc_id_elem.text.strip() if doc_id_elem is not None and doc_id_elem.text else 'unknown'
    for sent in doc.iter('sentence'):
        plain, spec, neg = extract_text_and_cues(sent)
        all_sentences.append({
            'sentence_id': sent.get('id'),
            'doc_id': doc_id,
            'text': plain,
            'epistemic_tag': 'UNCERTAIN' if spec else 'ASSERTIVE',
            'speculation_cues': spec,
            'negation_cues': neg,
            'token_count': len(plain.split())
        })

print(f'Total sentences: {len(all_sentences)}')
print(f'ASSERTIVE: {sum(1 for s in all_sentences if s["epistemic_tag"]=="ASSERTIVE")}')
print(f'UNCERTAIN: {sum(1 for s in all_sentences if s["epistemic_tag"]=="UNCERTAIN")}')

Total sentences: 11871
ASSERTIVE: 9770
UNCERTAIN: 2101


In [4]:
# Top hedge cues
all_cues = [c for s in all_sentences for c in s['speculation_cues']]
for cue, count in Counter(all_cues).most_common(15):
    print(f'  {cue}: {count}')

  may: 516
  suggest: 326
  suggesting: 150
  indicate that: 148
  or: 120
  whether: 96
  appears: 84
  might: 72
  suggested: 71
  could: 67
  suggests: 66
  likely: 60
  indicated that: 55
  indicating that: 55
  possible: 50


In [5]:
# Filter by paper criteria: 15-60 tokens, no anaphora
anaphoric = ['It ', 'This ', 'These ', 'That ', 'Those ', 'They ', 'Here ', 'Such ']
filtered = [s for s in all_sentences if 15 <= s['token_count'] <= 60 and not any(s['text'].startswith(w) for w in anaphoric)]

strong_cues = {'may','might','suggest','suggesting','suggests','suggested','could','whether','probably','possible','appears','appear','likely','indicate that','indicates that','putative','hypothesize','propose','proposes','proposed','seems'}

assertive_pool = [s for s in filtered if s['epistemic_tag'] == 'ASSERTIVE']
uncertain_pool = [s for s in filtered if s['epistemic_tag'] == 'UNCERTAIN' and any(c.lower() in strong_cues for c in s['speculation_cues'])]

random.shuffle(assertive_pool)
random.shuffle(uncertain_pool)

selected_source1 = uncertain_pool[:20] + assertive_pool[:20]
print(f'Source 1: {len(selected_source1)} sentences (20 UNCERTAIN + 20 ASSERTIVE)')

Source 1: 40 sentences (20 UNCERTAIN + 20 ASSERTIVE)


In [6]:
# Source 2: Fetch PubMed abstracts
import urllib.request, urllib.parse, time

BASE = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
queries = {
    'physics': ['quantum computing simulation', 'photonics optical fiber'],
    'computer_science': ['deep learning neural network', 'natural language processing transformer'],
    'engineering': ['nanotechnology nanoparticles synthesis', 'semiconductor device fabrication']
}

all_pmids = {}
for domain, qs in queries.items():
    for q in qs:
        params = urllib.parse.urlencode({'db':'pubmed','term':q,'retmax':15,'sort':'relevance','retmode':'json'})
        req = urllib.request.Request(f'{BASE}/esearch.fcgi?{params}', headers={'User-Agent':'ASL-Project/1.0'})
        with urllib.request.urlopen(req, timeout=15) as resp:
            ids = json.loads(resp.read().decode()).get('esearchresult',{}).get('idlist',[])
            for pid in ids:
                all_pmids[pid] = domain
        time.sleep(0.4)
print(f'PMIDs collected: {len(all_pmids)}')

# Fetch abstracts
pmid_list = list(all_pmids.keys())
all_abstracts = []
params = urllib.parse.urlencode({'db':'pubmed','id':','.join(pmid_list[:50]),'retmode':'xml','rettype':'abstract'})
req = urllib.request.Request(f'{BASE}/efetch.fcgi?{params}', headers={'User-Agent':'ASL-Project/1.0'})
with urllib.request.urlopen(req, timeout=30) as resp:
    aroot = ET.fromstring(resp.read().decode())
    for article in aroot.iter('PubmedArticle'):
        pmid = article.find('.//PMID').text
        parts = [ET.tostring(e, encoding='unicode', method='text').strip() for e in article.findall('.//AbstractText')]
        text = ' '.join(parts)
        if text and len(text) > 100:
            all_abstracts.append({'pmid': pmid, 'domain': all_pmids.get(pmid,'unknown'), 'abstract': text})
print(f'Abstracts fetched: {len(all_abstracts)}')

PMIDs collected: 90
Abstracts fetched: 44


In [7]:
# Split abstracts into sentences, find hedged ones
HEDGE = ['may','might','could','suggest','suggests','suggested','indicating','appear','appears','possible','probably','likely','propose','proposed','potential','hypothesize','whether']

def find_hedges(text):
    return [h for h in HEDGE if re.search(r'\b'+re.escape(h)+r'\b', text.lower())]

stem_sents = []
for ab in all_abstracts:
    for sent in re.split(r'(?<=[.!?])\s+(?=[A-Z])', ab['abstract']):
        tokens = len(sent.split())
        if 15 <= tokens <= 60 and not any(sent.startswith(w) for w in anaphoric):
            hedges = find_hedges(sent)
            stem_sents.append({'pmid':ab['pmid'],'domain':ab['domain'],'text':sent,'epistemic_tag':'UNCERTAIN' if hedges else 'ASSERTIVE','hedge_cues':hedges,'token_count':tokens})

unc2 = [s for s in stem_sents if s['epistemic_tag']=='UNCERTAIN'][:10]
ass2 = [s for s in stem_sents if s['epistemic_tag']=='ASSERTIVE'][:10]
selected_source2 = unc2 + ass2
print(f'Source 2: {len(selected_source2)} sentences')

Source 2: 20 sentences


In [8]:
# Combine into final 60-sentence dataset
all_stimuli = []
for idx, s in enumerate(selected_source1):
    all_stimuli.append({'stimulus_id':f'S{idx+1:03d}','source':'CoNLL-2010','doc_id':s['doc_id'],'text':s['text'],'epistemic_tag':s['epistemic_tag'],'epistemic_evidence':', '.join(s['speculation_cues']) if s['speculation_cues'] else 'N/A','token_count':s['token_count']})
for idx, s in enumerate(selected_source2):
    all_stimuli.append({'stimulus_id':f'S{41+idx:03d}','source':'PubMed_OA','doc_id':s['pmid'],'text':s['text'],'epistemic_tag':s['epistemic_tag'],'epistemic_evidence':', '.join(s['hedge_cues']) if s['hedge_cues'] else 'N/A','token_count':s['token_count']})

unc = sum(1 for s in all_stimuli if s['epistemic_tag']=='UNCERTAIN')
print(f'Final: {len(all_stimuli)} sentences | UNCERTAIN: {unc} | ASSERTIVE: {len(all_stimuli)-unc}')

with open('final_stimulus_dataset.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(all_stimuli[0].keys()))
    w.writeheader()
    w.writerows(all_stimuli)
print('Saved: final_stimulus_dataset.csv')
files.download('final_stimulus_dataset.csv')

Final: 60 sentences | UNCERTAIN: 30 | ASSERTIVE: 30
Saved: final_stimulus_dataset.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>